# 02 — Frozen DeBERTa-v3-base cache
Reuse `deberta.pt` when its fingerprint matches. Otherwise create it below. Previous two same-page OCR boxes + target; mean-pool target tokens only.


In [1]:
# Cell purpose: Load sequences and decide whether the frozen semantic feature cache is reusable.
# Inputs: sequences.json, the local DeBERTa path and feature settings.
# Outputs: A feature fingerprint and either the existing deberta.pt payload or a cache miss.

from pathlib import Path
import json
from config import CONFIG, ROOT, fingerprint, save_json, require_cache, run_stage, model_files

# Edit config.py once. Restart the kernel after changing it.
# Shared paths come from config.py; they are not hard-coded separately in each notebook.
CACHE = CONFIG['cache_dir']
RUN = CONFIG['run_dir']
CACHE.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
print('Existing caches:', [p.name for p in CACHE.iterdir() if p.is_file()])

import torch
# A missing prerequisite is created by its earlier notebook; an existing cache is reused.
require_cache(CACHE / 'sequences.json', '01_prepare_sequence_cache.ipynb')
sequences = json.loads((CACHE / 'sequences.json').read_text(encoding='utf-8'))
SEMANTICS = CACHE / 'deberta.pt'
# Fingerprint the inputs/settings so stale feature or sample caches are not silently reused.
signature = fingerprint({'sequences': sequences, 'model': CONFIG['deberta_model'],
                         'model_files': model_files(CONFIG['deberta_model']), 'max_tokens': CONFIG['max_tokens'], 'context': 2, 'pool': 'target_mean_v1'})
saved = torch.load(SEMANTICS, map_location='cpu', weights_only=True) if SEMANTICS.exists() else None
print('Reuse matching cache:', bool(saved and saved['signature'] == signature))

Existing caches: ['discovery.json', 'sequences.json']
Reuse matching cache: False


In [2]:
# Cell purpose: Create missing semantic features using previous two same-page OCR boxes plus the target.
# Inputs: OCR in existing box order and the locally downloaded frozen DeBERTa model.
# Outputs: One target-pooled 768-dimensional CPU vector per box in cache/deberta.pt.

# Only run the cache-building branch when the stored fingerprint is absent or stale.
if saved is None or saved['signature'] != signature:
    from transformers import AutoTokenizer, AutoModel
    from tqdm.auto import tqdm
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    # Use the downloaded tokenizer files only; do not download a replacement from the Hub.
    tokenizer = AutoTokenizer.from_pretrained(CONFIG['deberta_model'], local_files_only=True)
    tokenizer.padding_side = 'right'
    model = AutoModel.from_pretrained(CONFIG['deberta_model'], local_files_only=True).to(device).eval()
    model.requires_grad_(False)
    assert model.config.hidden_size == 768
    items, keys = [], []
    for seq in sequences:
        # Keep independent OCR histories for each page; context never crosses a page boundary.
        history = {0: [], 1: [], 2: []}
        # Process boxes in their saved order so every modality refers to the same sequence token.
        for box in seq['boxes']:
            # Use at most the two earlier boxes on this page; future boxes are excluded from DeBERTa input.
            previous = history[box['page_index']][-2:]
            context_ids = []
            for text in previous:
                context_ids += tokenizer.encode(text, add_special_tokens=False) + [tokenizer.sep_token_id]
            target_ids = tokenizer.encode(box['ocr'], add_special_tokens=False) or [tokenizer.unk_token_id]
            # Reserve CLS/SEP positions, keep the target first, then spend remaining tokens on recent context.
            budget = CONFIG['max_tokens'] - 2
            assert budget > 0
            target_ids = target_ids[-budget:]
            remaining = budget - len(target_ids)
            context_ids = context_ids[-remaining:] if remaining else []
            ids = [tokenizer.cls_token_id] + context_ids + target_ids + [tokenizer.sep_token_id]
            # This mask pools target-token states only; it is separate from encoder padding attention.
            target_mask = [0] * (1 + len(context_ids)) + [1] * len(target_ids) + [0]
            items.append((ids, target_mask))
            keys.append((seq['sequence_id'], box['box_id']))
            history[box['page_index']].append(box['ocr'])
    vectors = {}
    size = CONFIG['semantic_batch_size']
    # Disable autograd for frozen feature extraction or evaluation to reduce memory use.
    with torch.inference_mode():
        for start in tqdm(range(0, len(items), size), desc='DeBERTa'):
            batch = items[start:start+size]
            length = max(len(ids) for ids, mask in batch)
            ids = torch.full((len(batch), length), tokenizer.pad_token_id, dtype=torch.long, device=device)
            attention = torch.zeros_like(ids)
            target = torch.zeros_like(ids, dtype=torch.float)
            for i, (tokens, mask) in enumerate(batch):
                ids[i, :len(tokens)] = torch.tensor(tokens, device=device)
                attention[i, :len(tokens)] = 1
                target[i, :len(tokens)] = torch.tensor(mask, device=device)
            hidden = model(input_ids=ids, attention_mask=attention).last_hidden_state
            # Average target states only, producing one semantic vector per real text box.
            pooled = (hidden * target.unsqueeze(-1)).sum(1) / target.sum(1, keepdim=True)
            for key, vector in zip(keys[start:start+size], pooled.float().cpu()):
                vectors.setdefault(key[0], {})[key[1]] = vector.clone()
    saved = {'signature': signature, 'vectors': vectors}
    torch.save(saved, SEMANTICS)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
print('Semantic sequences:', len(saved['vectors']))

C:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\transformers\convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
C:\Users\ESHAAN\BYTE\godfather.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


DeBERTa:   0%|          | 0/137 [00:00<?, ?it/s]

Semantic sequences: 80


In [ ]:
# Cell purpose: Optional scratch cell; no operation is currently defined.
# Inputs: None.
# Outputs: None; this cell contains comments only.

